# Step 3 · Finding a pain → comfort direction and choosing a layer

**Pain–Comfort Lab · Looking inside a language model** · ⏱️ about 30 minutes · ⚡ GPU recommended

> ### 🤔 Wonder
> Is there a single arrow through 1,536-dimensional space that points from "pain" to "comfort"? And at which of the 28 layers is that arrow clearest?

**In this step you will:**

1. Build one **direction** per layer by averaging and subtracting, watching the shape after every line.
2. **Score** sentences by their "shadow" on that direction, first with a slow, readable loop, then with one fast line.
3. Grade the ruler with **accuracy** and **AUC**.
4. Pick the best layer using the **validation** set only, then freeze your choices.

## Setup and catch-up

These cells load the model and redo Step 2's snapshot collection with the library functions you built there.

In [ ]:
# 🔧 Setup: just run this cell. It works in Google Colab and in Jupyter on your own computer.
# It makes the lab's helper library (the `pclab` folder) and the sentence files available.
import subprocess, sys
from pathlib import Path
from importlib.metadata import version
from packaging.version import Version

REPO_URL = 'https://github.com/23menon23/pain-comfort-lab'   # the lab's home on GitHub

if Path('../pclab').exists():                    # you are inside a downloaded copy of the repository
    LAB_ROOT = Path('..').resolve()
else:                                            # Google Colab: download the lab once per session
    LAB_ROOT = Path('pain-comfort-lab').resolve()
    if not LAB_ROOT.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(LAB_ROOT)], check=True)
sys.path.insert(0, str(LAB_ROOT))

if Version(version('transformers')) < Version('4.57'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.57'], check=True)
    print('⚠️ Transformers was updated. Choose Runtime → Restart session, then run this cell again.')
else:
    import pclab
    from pclab import look, show_source
    print('✅ Ready. pclab', pclab.__version__, '| Transformers', version('transformers'))

In [ ]:
# 🔧 Load the model: just run this cell (the first download is about 3 GB and takes a few minutes).
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display

# Keep this model for class. 'Qwen/Qwen2.5-0.5B' uses less memory, but its results
# will differ, so record it as a different experiment.
MODEL_NAME = 'Qwen/Qwen2.5-1.5B'

SEED = pclab.set_seed(42)          # fixes random choices so results can be repeated
DEVICE = pclab.device_report()     # 'cuda' (GPU) or 'cpu'
tokenizer, model = pclab.load_model(MODEL_NAME)
LAYERS = pclab.get_layers(model)
N_LAYERS = len(LAYERS)
HIDDEN_SIZE = model.config.hidden_size
pclab.use_lab_style()
print(f"Loaded {pclab.model_facts(model)['model']}: {N_LAYERS} layers, {HIDDEN_SIZE} numbers per token")

In [ ]:
# 🔧 Catch-up: the sentences and snapshots from Step 2
sentences = pclab.load_sentences()
build_df, val_df = pclab.split(sentences, 'build'), pclab.split(sentences, 'validation')
train_act = pclab.collect(model, tokenizer, build_df['text'], 'Build snapshots')
val_act = pclab.collect(model, tokenizer, val_df['text'], 'Validation snapshots')
y_train, y_val = build_df['label'].to_numpy(), val_df['label'].to_numpy()
look(train_act, 'train_act', dims=('sentences', 'layers', 'numbers'))

## Part 1 · Build the direction, one line at a time

The recipe is the one you practised in Step 0 on invented 2-number points:

$$\text{direction} = \text{average comfort snapshot} - \text{average pain snapshot}$$

The only difference: we do it **for all 28 layers at once**, because our snapshots carry a layer dimension. Run each cell and check that the shape is what you expect.

**✏️ Predict first:** `train_act` is `(32, 28, 1536)`. What shape is the *average comfort snapshot*?

In [ ]:
# 🔧 Line 1: pick the comfort sentences with a mask, then average them
comfort_mask = y_train == 1
look(comfort_mask, 'comfort_mask', dims=('sentences',))

pos = train_act[comfort_mask].mean(axis=0)
look(pos, 'pos (average comfort snapshot)', dims=('layers', 'numbers'))

In [ ]:
# 🔧 Line 2: the same for pain, then subtract
neg = train_act[y_train == 0].mean(axis=0)
raw = pos - neg
look(raw, 'raw (the direction at every layer)', dims=('layers', 'numbers'))

In [ ]:
# 🔧 Line 3: how long is each layer's arrow?
lengths = np.linalg.norm(raw, axis=-1)
look(lengths, 'lengths', dims=('layers',))

In [ ]:
# 🔧 Line 4: rescale every arrow to length 1, and find the midpoints
unit = raw / np.maximum(lengths[:, None], 1e-8)
midpoints = (pos + neg) / 2
look(unit, 'unit', dims=('layers', 'numbers'))
print('Length of every unit arrow:', np.linalg.norm(unit, axis=-1).round(3)[:6], '...')

### 🔍 Code walk-through

| Code | Shape in → out | Why it's written this way |
|---|---|---|
| `y_train == 1` | `(32,)` → `(32,)` True/False | a **mask** that picks comfort sentences |
| `train_act[mask]` | `(32, 28, 1536)` → `(16, 28, 1536)` | keeps only the rows where the mask is True |
| `.mean(axis=0)` | `(16, 28, 1536)` → `(28, 1536)` | averages over **sentences** (axis 0), leaving one average snapshot per layer |
| `pos - neg` | `(28, 1536)` | 28 arrows, one per layer, each pointing from the pain average to the comfort average |
| `np.linalg.norm(raw, axis=-1)` | `(28, 1536)` → `(28,)` | the length of each arrow, measured along the last axis (the 1,536 numbers) |
| `lengths[:, None]` | `(28,)` → `(28, 1)` | adds a dimension of size 1 so each length lines up with **its own row**. See the 🛠️ below. |
| `np.maximum(..., 1e-8)` | same | a safety net: never divide by zero, even if an arrow had length 0 |
| `(pos + neg) / 2` | `(28, 1536)` | the midpoints: our ruler's zero mark at each layer |

> **🎛️ Why a length-1 ("unit") arrow?** Arrows at different layers have very different lengths (look at `lengths`, and remember the growing vectors from Step 1). A unit arrow is like a ruler with standard markings, so scores don't depend on how long the arrow happened to be.
>
> We call it a **candidate** direction because we don't yet know what it captures: physical sensation, positive-vs-negative words, or a mix. The rest of the lab helps find out.

### 🛠️ Tinker: what if we forget `[:, None]`?

**✏️ Predict first:** `raw` is `(28, 1536)` and `lengths` is `(28,)`. Will `raw / lengths` work? If it does, is it correct?

In [ ]:
# 🛠️ Try dividing without [:, None]
try:
    wrong = raw / lengths
    print('It ran! Shape:', wrong.shape, '... but is it right? Read the explanation below.')
except ValueError as error:
    print('ValueError:', error)

NumPy **broadcasting** lines shapes up **from the right**. It compares `1536` (last axis of `raw`) with `28` (the only axis of `lengths`). They don't match, so it fails. With `[:, None]`, `lengths` becomes `(28, 1)`: now the `28`s line up and the `1` stretches across all 1,536 numbers. Each row is divided by its own length.

*A sneaky case:* if the two sizes had happened to be equal (say a model with 28 numbers per token), the wrong version would have **run without an error** and silently divided each *column* by the wrong length. Shape checks catch bugs that error messages don't.

### Check against the library

`pclab.fit_directions` contains exactly these lines. Read it, then confirm it gives the same answer.

In [ ]:
show_source(pclab.fit_directions)
raw_vectors, unit_vectors, midpoints, vector_lengths = pclab.fit_directions(train_act, y_train)
print('\nSame as our step-by-step version:', np.allclose(unit_vectors, unit) and np.allclose(raw_vectors, raw))

## Part 2 · Scoring: each sentence's shadow on the ruler

Lay the unit arrow across the space with zero at the midpoint. Each sentence's snapshot casts a shadow onto it. Where the shadow lands is the sentence's **score**:

- **positive** → comfort side of the midpoint;
- **negative** → pain side;
- near zero → close to the dividing line.

The score is **not** a probability and **not** a measure of how much pain is described. It's a position on our ruler.

Researchers call a simple measuring tool like this a **probe**: something applied to a model's internal numbers to read out one kind of information. Our ruler is a probe.

### First, the slow and readable way: a loop

In [ ]:
# 🔧 Score every build sentence at every layer, one at a time
n_sentences = train_act.shape[0]
scores_loop = np.zeros((n_sentences, N_LAYERS))          # an empty grid to fill in: sentences × layers

for n in range(n_sentences):
    for layer in range(N_LAYERS):
        shifted = train_act[n, layer] - midpoints[layer]                 # move the zero mark to the midpoint
        scores_loop[n, layer] = np.dot(shifted, unit_vectors[layer])    # the shadow on the ruler

look(scores_loop, 'scores_loop', dims=('sentences', 'layers'))

### Now the fast way: one line with `einsum`

`np.einsum` describes a calculation with letters, one letter per axis:

```python
np.einsum('nld,ld->nl', shifted_activations, unit_vectors)
```

| Letters | Meaning |
|---|---|
| `nld` | the first input has axes **n**(sentences), **l**(layers), **d**(numbers) |
| `ld` | the second input has axes **l**(layers), **d**(numbers) |
| `->nl` | the output keeps **n** and **l** |
| `d` disappears | so NumPy multiplies along `d` and **adds up**: that's the dot product |
| `l` appears in both inputs | so each sentence's layer `l` is paired with layer `l`'s arrow |

It's exactly the double loop above, written as a recipe.

In [ ]:
# 🔧 The library's one-line version, checked against the loop
show_source(pclab.scores_at_all_layers)
train_scores = pclab.scores_at_all_layers(train_act, unit_vectors, midpoints)
print('\nSame as the loop:', np.allclose(train_scores, scores_loop, rtol=1e-4, atol=1e-3))

`midpoints[None, :, :]` in that function turns the midpoints from `(28, 1536)` into `(1, 28, 1536)`, so the same midpoints are subtracted from **every** sentence.

### 📦 Look at the scores for one layer

Here's every build sentence, sorted by its score at the middle layer.

**✏️ Predict first:** will the build sentences be perfectly separated (all pain negative, all comfort positive)? Is that impressive?

In [ ]:
# 🧪 Experiment: change `layer_to_view` (0 = first layer)
layer_to_view = N_LAYERS // 2 - 1      # Python index: for 28 layers this is layer 14, as in Step 1
view = build_df[['text', 'category']].assign(score=train_scores[:, layer_to_view]).sort_values('score')
with pd.option_context('display.max_colwidth', 80):
    display(view)

The build set is usually separated well, but that's **grading your own homework**: the ruler was built from these very sentences. That's why we need the validation and test sets.

## Part 3 · Two ways to grade a ruler

- **Accuracy:** the fraction of sentences on the correct side of zero. With equal numbers of pain and comfort sentences, guessing gets 50%.
- **AUC:** pick one comfort sentence and one pain sentence at random. How often does the comfort one score higher? 0.5 = coin flip, 1.0 = always right. AUC checks the **ordering** of scores, without needing a dividing line.

A tiny example you can check by hand: comfort scores `[0.9, 0.2]`, pain scores `[0.5, -0.3]`. There are 2 × 2 = 4 comfort–pain pairs. In how many does comfort score higher?

In [ ]:
# 🔧 AUC by hand vs. scikit-learn
from sklearn.metrics import accuracy_score, roc_auc_score
comfort_scores, pain_scores = [0.9, 0.2], [0.5, -0.3]
wins = sum(c > p for c in comfort_scores for p in pain_scores)
print('Comfort wins', wins, 'of 4 pairs → AUC =', wins / 4)

labels = [1, 1, 0, 0]
print('scikit-learn agrees:', roc_auc_score(labels, comfort_scores + pain_scores))
print('Accuracy with zero as the dividing line:', accuracy_score(labels, [s > 0 for s in comfort_scores + pain_scores]))

Both grades happen to be 0.75 here, but for **different reasons**. Accuracy loses a point because the pain score 0.5 lands on the comfort side of zero. AUC loses a point because one pair (comfort 0.2, pain 0.5) is in the wrong order.

**🛠️ Tinker:** subtract 1.0 from every score in the cell above and rerun. Accuracy changes, but AUC doesn't, because AUC only cares about the *order* of the scores, not where zero is. That's why we report both.

## Part 4 · Choose a layer, using the validation set only

We score the **validation** sentences at all layers and pick the layer with the best validation accuracy. Ties go to higher AUC, then the earlier layer. This rule is decided **before** we look at the test set.

**✏️ Predict first:** will the best layer be early, middle, or late? Why?

In [ ]:
# 🔧 Score the validation set at every layer and choose
val_scores = pclab.scores_at_all_layers(val_act, unit_vectors, midpoints)
validation_results = pclab.layer_table(val_scores, y_val)
build_results = pclab.layer_table(train_scores, y_train)
BEST_LAYER = pclab.choose_layer(validation_results)       # a 0-based Python index

C = pclab.COLORS
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(build_results['layer'], build_results['accuracy'], ':', color=C['neutral'], label='Build accuracy (own homework)')
ax.plot(validation_results['layer'], validation_results['accuracy'], 'o-', color=C['comfort'], label='Validation accuracy')
ax.plot(validation_results['layer'], validation_results['auc'], 's--', color=C['pain'], markersize=4, label='Validation AUC')
ax.axhline(0.5, color='gray', linewidth=1, label='Chance (50%)')
ax.axvline(BEST_LAYER + 1, color=C['accent'], linestyle='--', label=f'Selected layer {BEST_LAYER + 1}')
ax.set(xlabel='Layer', ylabel='Score', ylim=(0, 1.05), title='Validation results: choose a layer before opening the test set')
ax.legend(loc='lower right', fontsize=8)
plt.show()

print(f'Selected layer: {BEST_LAYER + 1} (Python index {BEST_LAYER})')
display(validation_results.sort_values(['accuracy', 'auc', 'layer'], ascending=[False, False, True]).head(5))

### 🔍 Code walk-through: how the layer is chosen

In [ ]:
show_source(pclab.layer_table)
print()
show_source(pclab.choose_layer)

| Code | What it does | Why |
|---|---|---|
| `(scores[:, i] > 0).astype(int)` | turns each score into a prediction: 1 (comfort) if positive, else 0 | zero is the midpoint, our dividing line |
| `[... for i in range(n_layers)]` | repeats the grading for every layer | a **list comprehension**: a compact `for` loop that builds a list |
| `pd.DataFrame({...})` | a table with one row per layer | 🧱 easy to sort, plot and save |
| `sort_values([...], ascending=[False, False, True])` | sorts by accuracy (high first), then AUC (high first), then layer (low first) | encodes the tie-break rule exactly, so nobody can "choose" a nicer layer later |
| `int(...) - 1` | converts the human layer number to a Python index | layers are shown as 1–28 but stored as 0–27 |

### What this chart does and doesn't mean

The selected layer is simply where **our ruler worked best on our 12 validation sentences**. It is **not** "where the model feels pain" or "where it understands pain".

- With only 12 sentences, **one mistake moves accuracy by about 8 percentage points**, so small bumps are probably noise.
- Several layers might tie; the tie-break picks one, and that doesn't make it special.
- Early layers can score well just by noticing obvious words.

**✏️ Explain:** describe your chart in one sentence, without using the words "feels" or "conscious".

### 🧊 Freeze your choices

From here on, the direction, the midpoint and the chosen layer are **frozen**. No switching to a better-looking layer after seeing the test results. Write your selected layer here: **Layer ____**

Later notebooks recompute these with `pclab.rebuild_lab_state(...)`, which follows the same rule, so they'll arrive at the same layer.

## 🌱 Optional: how many sentence pairs does the ruler need?

The build set has 16 pairs. Would 2 be enough? This cell rebuilds the ruler from only the first *k* pairs and grades it on validation at your chosen layer. (This is exploration, not part of the frozen result.)

**✏️ Predict first:** at what *k* does validation accuracy stop improving?

In [ ]:
# 🧪 Experiment: change the list of k values
rows = []
for k in [1, 2, 4, 8, 16]:
    keep = slice(0, 2 * k)                      # the first k pairs = the first 2k rows
    _, u, m, _ = pclab.fit_directions(train_act[keep], y_train[keep])
    s = pclab.scores_at_all_layers(val_act, u, m)[:, BEST_LAYER]
    rows.append({'pairs_used': k, 'validation_accuracy': accuracy_score(y_val, s > 0),
                 'validation_auc': roc_auc_score(y_val, s)})
display(pd.DataFrame(rows))

Results from tiny samples are noisy: one lucky or unlucky pair can swing them. A real study would repeat this with many different random choices of *k* pairs. (🌱 That's a good Step 7 investigation.)

---

## 🧭 Explain it

1. In your own words, how is the direction built? What shape does it have, and why?
2. What does a positive score mean? What does it **not** mean?
3. Why don't we grade the ruler on the build set? Use the chart to explain.
4. Why must the layer be chosen *before* opening the test set?

## 🌱 Curiosity branches

- **Related concepts:** build a hot–cold or rough–smooth direction from your own sentence pairs. Compare it with the pain–comfort arrow using cosine similarity (Step 2's `cosine` function). Same arrow or a different one?
- **Template test:** rebuild everything with a different measurement template. Does the chosen layer move?
- **Relative depth:** if you switch to `Qwen/Qwen2.5-0.5B` (24 layers), does the best layer sit at a similar *fraction* of the way through?

**Next:** [Step 4 · The final exam and tricky sentences](04_final_exam_and_tricky_sentences.ipynb)